# 🧹 Data Cleaning, Normalization & Star Schema Preparation### Retail Business Intelligence & Customer Analytics Platform**Author:** Senior Analytics Engineer**Objective:** Standardize raw transactional data, enforce integrity constraints, engineer initial analytical flags, and partition data into Star Schema dimensional entities.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_path = Path('../data/raw/superstore_raw.csv')
cleaned_path = Path('../data/cleaned/superstore_cleaned.csv')

df_raw = pd.read_csv(raw_path, encoding='utf-8')
print(f"Loaded raw records: {len(df_raw):,}")

## 1. Column Standardization to Snake Case

In [ ]:
column_map = {
    "Row ID": "row_id",
    "Order ID": "order_id",
    "Order Date": "order_date",
    "Ship Date": "ship_date",
    "Ship Mode": "ship_mode",
    "Customer ID": "customer_id",
    "Customer Name": "customer_name",
    "Segment": "segment",
    "Country": "country",
    "City": "city",
    "State": "state",
    "Postal Code": "postal_code",
    "Region": "region",
    "Product ID": "product_id",
    "Category": "category",
    "Sub-Category": "sub_category",
    "Product Name": "product_name",
    "Sales": "sales",
    "Quantity": "quantity",
    "Discount": "discount",
    "Profit": "profit"
}
df_clean = df_raw.rename(columns=column_map)
df_clean.columns.tolist()

## 2. Type Conversion & String Normalization

In [ ]:
# Convert Dates
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'], format='mixed')
df_clean['ship_date'] = pd.to_datetime(df_clean['ship_date'], format='mixed')

# Clean Strings
string_cols = df_clean.select_dtypes(include='object').columns
for col in string_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Format Postal Code (US 5-digit zip with zero-padding)
df_clean['postal_code'] = df_clean['postal_code'].astype(str).str.replace('.0', '', regex=False).str.zfill(5)

# Round financials
df_clean['sales'] = df_clean['sales'].astype(float).round(4)
df_clean['discount'] = df_clean['discount'].astype(float).round(4)
df_clean['profit'] = df_clean['profit'].astype(float).round(4)
df_clean['quantity'] = df_clean['quantity'].astype(int)

df_clean.info()

## 3. Data Quality Assertions & Export

In [ ]:
# Assert integrity rules
assert (df_clean['sales'] >= 0).all(), "Negative sales found!"
assert (df_clean['quantity'] > 0).all(), "Invalid quantity found!"
assert (df_clean['ship_date'] >= df_clean['order_date']).all(), "Invalid shipping dates found!"

cleaned_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(cleaned_path, index=False)
print(f"Cleaned dataset successfully exported to {cleaned_path}")

## 4. Dimensional Modeling SummaryThe cleaned transaction dataset is prepared for transformation into:- `fact_sales` (Grain: 1 row per order line item)- `dim_customer` (Customer profile & lifetime spending metrics)- `dim_product` (Catalog hierarchy: Category -> Sub-Category -> Product)- `dim_geography` (Geographic hierarchy: Country -> Region -> State -> City -> Postal Code)- `dim_ship_mode` (Logistics fulfillment methods)- `dim_date` (Complete analytical calendar table)